In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import *
from pyspark.sql.window import *

spark = SparkSession.builder.appName("PharmaceuticalEquipment").getOrCreate()

# Create df1
df1_data = [
    ("EQ001", "Mixer", "2020-01-01"),
    ("EQ002", "Centrifuge", "2020-02-01"),
    ("EQ003", "Pipette", "2020-03-01"),
]

df1_schema = StructType(
    [
        StructField("equipment_id", StringType(), True),
        StructField("equipment_name", StringType(), True),
        StructField("purchase_date", StringType(), True),
    ]
)

df1 = spark.createDataFrame(df1_data, schema=df1_schema)

# Create df2
df2_data = [
    ("EQ001", "2021-06-01", 500.0),
    ("EQ002", "2021-07-01", 400.0),
    ("EQ001", "2021-07-02", 600.0),
]

df2_schema = StructType(
    [
        StructField("equipment_id", StringType(), True),
        StructField("maintenance_date", StringType(), True),
        StructField("maintenance_cost", DoubleType(), True),
    ]
)

df2 = spark.createDataFrame(df2_data, schema=df2_schema)

In [6]:
df2_grp = df2.groupBy(col("equipment_id")).agg(
    max("maintenance_date").alias("latest_maintenance_date")
)

In [11]:
window_spec = Window.partitionBy("equipment_id").orderBy("equipment_id")
df1.join(df2_grp, "equipment_id").withColumn(
    "maintenance_cost_rank", row_number().over(window_spec)
).show()

+------------+--------------+-------------+-----------------------+---------------------+
|equipment_id|equipment_name|purchase_date|latest_maintenance_date|maintenance_cost_rank|
+------------+--------------+-------------+-----------------------+---------------------+
|       EQ001|         Mixer|   2020-01-01|             2021-07-02|                    1|
|       EQ002|    Centrifuge|   2020-02-01|             2021-07-01|                    1|
+------------+--------------+-------------+-----------------------+---------------------+

